In [1]:
import pandas as pd
import numpy as np

print("正在讀取正確路徑的檔案...")
train_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip')
test_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip')

print("=== 資料讀取成功！ ===")
print(f"訓練集（Train）大小: {train_df.shape} 筆")
print(f"測試集（Test）大小: {test_df.shape} 筆\n")

正在讀取正確路徑的檔案...
=== 資料讀取成功！ ===
訓練集（Train）大小: (159571, 8) 筆
測試集（Test）大小: (153164, 2) 筆



In [2]:
import os

print("---所有檔案路徑 ---")
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            print(os.path.join(root, file))
else:
    print("找不到資料夾！")
    print("目前的檔案有：", os.listdir('.'))
    

---所有檔案路徑 ---
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


In [3]:
import pandas as pd
train_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip')
test_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip')

print("=== 資料讀取成功！ ===")
print(f"訓練集（Train）大小: {train_df.shape} 筆")
print(f"測試集（Test）大小: {test_df.shape} 筆\n")

print("--- 訓練集前 3 筆資料預覽 ---")
# 顯示 id, comment_text 和 6 個標籤
print(train_df[['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].head(3))

=== 資料讀取成功！ ===
訓練集（Train）大小: (159571, 8) 筆
測試集（Test）大小: (153164, 2) 筆

--- 訓練集前 3 筆資料預覽 ---
                                        comment_text  toxic  severe_toxic  \
0  Explanation\nWhy the edits made under my usern...      0             0   
1  D'aww! He matches this background colour I'm s...      0             0   
2  Hey man, I'm really not trying to edit war. It...      0             0   

   obscene  threat  insult  identity_hate  
0        0       0       0              0  
1        0       0       0              0  
2        0       0       0              0  


In [4]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

label_sums = train_df[labels].sum()

print("=== 各類別有毒留言數量 ===")
print(label_sums)

print("\n=== 各類別佔總資料比例 (%) ===")
print((label_sums / len(train_df)) * 100)

=== 各類別有毒留言數量 ===
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

=== 各類別佔總資料比例 (%) ===
toxic            9.584448
severe_toxic     0.999555
obscene          5.294822
threat           0.299553
insult           4.936361
identity_hate    0.880486
dtype: float64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

train_df['comment_text'] = train_df['comment_text'].fillna(" ")
test_df['comment_text'] = test_df['comment_text'].fillna(" ")
X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_df['comment_text'], 
    train_df[labels], 
    test_size=0.2, 
    random_state=42
)

print("正在將文字轉換為 TF-IDF 特徵... (這需要幾秒鐘)")
vectorizer = TfidfVectorizer(max_features=30000, stop_words='english', ngram_range=(1, 2))

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(test_df['comment_text'])

submission = pd.DataFrame({'id': test_df['id']})
val_auc_scores = []

print("\n--- 開始訓練 6 個標籤的邏輯迴歸模型 ---")
for label in labels:
    model = LogisticRegression(C=4.0, max_iter=500, solver='liblinear')
    model.fit(X_train, y_train[label])
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val[label], val_preds)
    val_auc_scores.append(auc)
    print(f"標籤 [{label:13}] 的驗證集 AUC: {auc:.4f}")
    submission[label] = model.predict_proba(X_test)[:, 1]

print(f"\n平均驗證集 AUC (Mean AUC): {np.mean(val_auc_scores):.4f}")
submission.to_csv('submission.csv', index=False)
print("\n基準模型預測完成！檔案已儲存為 'submission.csv'")


正在將文字轉換為 TF-IDF 特徵... (這需要幾秒鐘)

--- 開始訓練 6 個標籤的邏輯迴歸模型 ---
標籤 [toxic        ] 的驗證集 AUC: 0.9683
標籤 [severe_toxic ] 的驗證集 AUC: 0.9798
標籤 [obscene      ] 的驗證集 AUC: 0.9818
標籤 [threat       ] 的驗證集 AUC: 0.9871
標籤 [insult       ] 的驗證集 AUC: 0.9728
標籤 [identity_hate] 的驗證集 AUC: 0.9670

平均驗證集 AUC (Mean AUC): 0.9761

基準模型預測完成！檔案已儲存為 'submission.csv'


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 1. 告訴 W&B 你的金鑰（請在引號內填入你真正的金鑰密碼）
os.environ["WANDB_API_KEY"] = "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"

# 2. 徹底封口！下令後台絕對不准跳出任何需要人類互動的點選提示
os.environ["WANDB_SILENT"] = "true"

# 以下是原本的工具箱與程式碼
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

# 確保資料讀取
print("正在從正確路徑重新讀取資料...")
train_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip')
test_df = pd.read_csv('/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip')
print("資料讀取成功！")

# 1. 準備資料與標籤
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
train_df['comment_text'] = train_df['comment_text'].fillna(" ")
test_df['comment_text'] = test_df['comment_text'].fillna(" ")

# 切分訓練集與驗證集 (10% 驗證，90% 訓練)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['comment_text'].values, 
    train_df[labels].values, 
    test_size=0.1, 
    random_state=42
)

# 2. 載入 DistilBERT 的 Tokenizer
print("正在載入 Tokenizer...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. 建立 PyTorch Dataset 格式
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=192):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True
        )
        
        item = {
            'input_ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(inputs['attention_mask'], dtype=torch.long)
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
            
        return item

print("正在轉換資料格式...")
train_dataset = ToxicDataset(train_texts, train_labels, tokenizer)
val_dataset = ToxicDataset(val_texts, val_labels, tokenizer)
test_dataset = ToxicDataset(test_df['comment_text'].values, None, tokenizer)

# 4. 載入預訓練模型
print("正在載入 DistilBERT 模型...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=6, 
    problem_type="multi_label_classification"
)

# 5. 設定指標計算函數
labels_list = labels
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    probs = 1 / (1 + np.exp(-logits))
    
    auc_scores = {}
    for i, label_name in enumerate(labels_list):
        try:
            auc_scores[f"auc_{label_name}"] = roc_auc_score(labels[:, i], probs[:, i])
        except:
            auc_scores[f"auc_{label_name}"] = 0.5
            
    auc_scores["mean_auc"] = np.mean([auc_scores[f"auc_{lbl}"] for lbl in labels_list])
    return auc_scores

# 6. 設定訓練參數與 W&B 連動
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,              
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=64,
    warmup_steps=500,                
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="steps",       
    eval_steps=500,              
    save_steps=500,
    report_to="wandb",               
    run_name="distilbert_toxic_run", 
    fp16=True                        
)

# 7. 初始化 Trainer 並開始訓練
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\n--- 🚀 開始微調 Transformer 模型 (需要大約 20-30 分鐘) ---")
trainer.train();

# 8. 預測測試集並產生預測結果
print("\n--- 正在產生測試集預測結果 ---")
predictions = trainer.predict(test_dataset)
test_probs = 1 / (1 + np.exp(-predictions.predictions))

# 儲存成全新的 submission.csv
submission_transformer = pd.DataFrame({'id': test_df['id']})
for i, label in enumerate(labels):
    submission_transformer[label] = test_probs[:, i]

submission_transformer.to_csv('submission.csv', index=False)
print("Transformer 預測完成！新檔案已覆蓋儲存為 'submission.csv'")

正在從正確路徑重新讀取資料...
資料讀取成功！
正在載入 Tokenizer...
正在轉換資料格式...
正在載入 DistilBERT 模型...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



--- 🚀 開始微調 Transformer 模型 (需要大約 20-30 分鐘) ---


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- 正在產生測試集預測結果 ---


Transformer 預測完成！新檔案已覆蓋儲存為 'submission.csv'
